Copia histórica sanitizada. No ejecutar para el flujo nuevo: usa EmotiveAD_Colab.ipynb.
Se eliminaron credenciales y salidas; el código histórico conserva sus incompatibilidades.

In [ ]:
import os

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  Radar — 16 Emociones × 3 Locutores + Promedio
#  Output: (1) Combined 2×8 grid  (2) 16 PNGs individuales
#
#  COLAB: upload the 3 CSVs, adjust the 3 paths, run
# ═══════════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import os

# ── Paths ────────────────────────────────────────────────────────────────────
PATH_ANDREA    = "AudioFeaturesOS_Andrea.csv"
PATH_SERGIO    = "AudioFeaturesOS_Sergio.csv"
PATH_SEBASTIAN = "AudioFeaturesOS_Sebastian.csv"
OUTPUT_DIR     = "radar_individual"

# ── Colors ───────────────────────────────────────────────────────────────────
C = {
    "Andrea":    "#7B5EA7",   # purple
    "Sergio":    "#E8A838",   # mustard
    "Sebastian": "#3A86D4",   # cornflower blue
    "Promedio":  "#999999",   # grey average
}

# ── Plutchik: primary → (low variant, high variant) ──────────────────────────
PLUTCHIK_MAP = {
    "Joy":          ("serenidad",            "éxtasis"),
    "Trust":        ("aceptación",           "admiración"),
    "Fear":         ("aprensión",            "terror"),
    "Surprise":     ("distracción",          "asombro"),
    "Sadness":      ("pensamiento profundo", "duelo"),
    "Disgust":      ("aburrimiento",         "aversión"),
    "Anger":        ("molestia",             "furia"),
    "Anticipation": ("interés",              "vigilancia"),
}
PRIMARIES = list(PLUTCHIK_MAP.keys())
FEATURES  = ["Intensidad", "Entonación", "Duración", "Ritmo"]
LABELS    = ["I", "T", "D", "R"]
ANGLES    = np.linspace(0, 2*np.pi, 4, endpoint=False).tolist()
ANGLES   += [ANGLES[0]]

# ── Load CSVs ────────────────────────────────────────────────────────────────
def load(path, name):
    df = pd.read_csv(path)
    df["_key"] = df["Emoción complementaria"].str.strip().str.lower()
    return df

dfs = {}
for path, name in [(PATH_ANDREA,"Andrea"),(PATH_SERGIO,"Sergio"),(PATH_SEBASTIAN,"Sebastian")]:
    try:
        dfs[name] = load(path, name)
        print(f"✅ {name}: {len(dfs[name])} rows loaded")
    except FileNotFoundError:
        print(f"⚠️  {name}: not found at '{path}'")

assert dfs, "No CSV files loaded — check paths"

# ── Helpers ──────────────────────────────────────────────────────────────────
def get_vals(df, key):
    row = df[df["_key"] == key.strip().lower()]
    return row[FEATURES].values[0].tolist() if not row.empty else None

def style_ax(ax, fs=28):
    ax.set_facecolor("#f9f9f9")
    ax.patch.set_facecolor("#f9f9f9")
    ax.yaxis.grid(True, linestyle=(0,(4,6)), linewidth=1, color="#717171")
    ax.xaxis.grid(True, linestyle=(0,(4,6)), linewidth=1, color="#717171")
    ax.spines["polar"].set_visible(True)
    ax.spines["polar"].set_linewidth(1.0)
    ax.spines["polar"].set_linestyle((0,(4,6)))
    ax.spines["polar"].set_edgecolor("#717171")
    ax.set_xticks(ANGLES[:-1])
    ax.set_xticklabels(LABELS, fontsize=fs)
    ax.set_yticks([1, 2, 3, 4, 5])
    ax.set_yticklabels(["1","2","3","4","5"], fontsize=fs-4)
    ax.set_ylim(0, 5)
    ax.set_rlabel_position(45)   # numbers at 45° — between T and I axes

def draw(ax, var_key, fs=28):
    style_ax(ax, fs)
    all_vals = []
    for name, df in dfs.items():
        v = get_vals(df, var_key)
        if v is None: continue
        all_vals.append(v)
        ax.plot(ANGLES, v+[v[0]], lw=2, color=C[name], alpha=0.92)
        ax.fill(ANGLES, v+[v[0]], alpha=0.3, color=C[name])
    if all_vals:
        avg = np.mean(all_vals, axis=0).tolist()
        ax.plot(ANGLES, avg+[avg[0]], lw=2.5, ls="--", color=C["Promedio"], alpha=0.85, zorder=5)
        ax.fill(ANGLES, avg+[avg[0]], alpha=0.1, color=C["Promedio"])

LEGEND_HANDLES = [
    mpatches.Patch(color=C["Andrea"],    label="And"),
    mpatches.Patch(color=C["Sergio"],    label="Ser"),
    mpatches.Patch(color=C["Sebastian"], label="Seb"),
    Line2D([0],[0], color=C["Promedio"], label="Av"),
]
import matplotlib.font_manager as fm
import requests, zipfile, io, os, matplotlib
import matplotlib.pyplot as plt

# Descargar Inter
font_url = "https://github.com/rsms/inter/releases/download/v4.0/Inter-4.0.zip"
r = requests.get(font_url)
z = zipfile.ZipFile(io.BytesIO(r.content))

font_dir = os.path.expanduser("~/.fonts/inter")
os.makedirs(font_dir, exist_ok=True)

# Extraer solo los .ttf estáticos (no variables)
for name in z.namelist():
    if name.endswith(".ttf") and "InterVariable" not in name:
        data = z.read(name)
        fpath = os.path.join(font_dir, os.path.basename(name))
        with open(fpath, "wb") as f:
            f.write(data)

# Registrar cada archivo individualmente
for ttf in os.listdir(font_dir):
    if ttf.endswith(".ttf"):
        fm.fontManager.addfont(os.path.join(font_dir, ttf))

# Rebuild cache y aplicar
fm._load_fontmanager(try_read_cache=False)
plt.rcParams["font.family"]     = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Inter", "DejaVu Sans"]
print("✅ Inter font ready")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — 16 individual PNGs
# All labels placed with fig.text() to avoid polar axis overlap
# ═══════════════════════════════════════════════════════════════════════════════
os.makedirs(OUTPUT_DIR, exist_ok=True)

for primary in PRIMARIES:
    low_var, high_var = PLUTCHIK_MAP[primary]
    for var_key, intensity in [(high_var,"High"),(low_var,"Low")]:

        fig_i, ax_i = plt.subplots(1, 1, figsize=(8, 9),
                                   subplot_kw=dict(polar=True))
        fig_i.patch.set_facecolor("white")
        fig_i.subplots_adjust(top=0.72)   # leave 28% at top for labels

        draw(ax_i, var_key, fs=28)

        # Line 1: Primary · Intensity (italic, small)
        fig_i.text(0.5, 0.97, f"{primary}  ·  {intensity} Intensity",
                   ha="center", va="top", fontsize=20,
                   color="#666666", style="italic")

        # Line 2: Complementary emotion name (bold, large)
        fig_i.text(0.5, 0.91, var_key.capitalize(),
                   ha="center", va="top", fontsize=34,
                   fontweight="bold", color="#222222")

        # Line 3: Narrator colored labels
        narrators = list(dfs.keys())
        spacing = 0.28
        base_x = 0.5 - spacing * (len(narrators) - 1) / 2
        for i, name in enumerate(narrators):
            fig_i.text(base_x + i * spacing, 0.84,
                       f"• {name}", ha="center", va="top",
                       fontsize=24, color=C[name])

        # Line 4: Average
        fig_i.text(0.5, 0.78, "  ",
                   ha="center", va="top", fontsize=20, color=C["Promedio"])

        fname = f"{OUTPUT_DIR}/{intensity}_{primary}_{var_key.capitalize()}.png"
        plt.savefig(fname, dpi=150, bbox_inches="tight", facecolor="white")
        plt.close()
        print(f"  ✅ {fname}")

print(f"\n✅ 16 individual charts saved to '{OUTPUT_DIR}/'")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Combined 2×8 grid
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 8, figsize=(56, 18),
                         subplot_kw=dict(polar=True))
fig.patch.set_facecolor("white")
fig.subplots_adjust(top=0.85, bottom=0.08, wspace=0.35, hspace=0.8)

for col, primary in enumerate(PRIMARIES):
    low_var, high_var = PLUTCHIK_MAP[primary]
    for row, (var_key, int_label) in enumerate([(high_var,"HIGH"),(low_var,"LOW")]):
        ax = axes[row, col]
        draw(ax, var_key, fs=20)

        pos = ax.get_position()
        cx  = pos.x0 + pos.width / 2

        # Complementary emotion name (above each subplot)
        fig.text(cx, pos.y1 + 0.055, var_key.capitalize(),
                 ha="center", va="bottom", fontsize=22,
                 fontweight="bold", color="#222222")
        # Primary name (only row 0, higher up)
        if row == 0:
            fig.text(cx, pos.y1 + 0.105, primary,
                     ha="center", va="bottom", fontsize=24,
                     fontweight="bold", color="#3A3A3A")
        # Row intensity label (leftmost column only)
        if col == 0:
            rl = "HIGH INTENSITY" if row == 0 else "LOW INTENSITY"
            fig.text(pos.x0 - 0.025, pos.y0 + pos.height/2,
                     rl, ha="right", va="center",
                     fontsize=17, fontweight="bold",
                     color="#555555", rotation=90)

fig.legend(handles=LEGEND_HANDLES, loc="lower center", ncol=4,
           fontsize=22, frameon=True, bbox_to_anchor=(0.5, 0.01),
           edgecolor="#cccccc", fancybox=True)
fig.suptitle("Prosodic Profiles — 16 Emotional Renditions · 3 Narrators",
             fontsize=28, fontweight="bold", y=0.99, color="#1a1a1a")

plt.savefig("Radar_16Emotions_Combined.png", dpi=150,
            bbox_inches="tight", facecolor="white")
plt.show()
print("✅ Radar_16Emotions_Combined.png saved")

In [ ]:
import shutil
from google.colab import files

# Name of the directory to zip (from the previous cell's OUTPUT_DIR)
directory_to_zip = OUTPUT_DIR

# Name of the output zip file
zip_filename = f"{directory_to_zip}.zip"

# Create the zip archive
shutil.make_archive(directory_to_zip, 'zip', directory_to_zip)

# Download the zip file
files.download(zip_filename)

print(f"✅ Successfully created and downloaded '{zip_filename}'")